# Sentiment Model Comparison — Reproduce Table

Loads prediction files from each approach and recomputes **Accuracy**, **Macro F1**, and **Neg F1** (F1 for the negative/minority class).

All approaches use the same guitar test set (80/20 stratified split, `random_state=42`).

In [9]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

In [10]:
def metrics(y_true, y_pred):
    return {
        'Acc':    round(accuracy_score(y_true, y_pred), 3),
        'Mac F1': round(f1_score(y_true, y_pred, average='macro'), 3),
        'Neg F1': round(f1_score(y_true, y_pred, pos_label=0, average='binary'), 3),
    }

results = {}

## TF-IDF + LR

In [11]:
df = pd.read_parquet('tf_idf - lr/predictions.parquet')
results['TF-IDF + LR'] = metrics(df['label'], df['pred'])

## Sentence Transformer + LR

In [12]:
df = pd.read_parquet('sentence_transformer - lr/predictions.parquet')
results['SentTransf. + LR'] = metrics(df['label'], df['pred'])

## Pre-trained BERT (nlptown & CardiffNLP)

In [13]:
df = pd.read_parquet('pretrained_bert/predictions.parquet')
results['nlptown BERT']       = metrics(df['label'], df['pred_nlptown'])
results['CardiffNLP RoBERTa'] = metrics(df['label'], df['pred_cardiffnlp'])

## Zero-shot (BART & DeBERTa)

In [14]:
df = pd.read_parquet('zero_shot/predictions.parquet')
results['BART zero-shot']    = metrics(df['label'], df['pred_bart'])
results['DeBERTa zero-shot'] = metrics(df['label'], df['pred_deberta'])

## Fine-tuned RoBERTa

`bert_predictions.csv` covers the full guitar dataset (inference on all reviews). We reconstruct the same test split used during training to isolate held-out predictions.

In [15]:
df = pd.read_csv('fine_tuned_roberta/bert_predictions.csv', low_memory=False)

# Keep only binary-labelled rows (drops rating=3)
df_bin = df[df['label'].isin([0.0, 1.0])].copy()
df_bin['label'] = df_bin['label'].astype(int)

# Reconstruct the exact same test split (test_size=0.2, random_state=42, stratified)
_, test_df = train_test_split(df_bin, test_size=0.2, random_state=42, stratify=df_bin['label'])

results['Fine-tuned RoBERTa'] = metrics(test_df['label'], test_df['pred_label'])

## Mistral-7B 5-shot

In [16]:
df = pd.read_csv('llm_few_shot_doc/llm_predictions.csv', low_memory=False)

label_map = {'positive': 1, 'negative': 0}
y_true = df['sentiment'].map(label_map)
y_pred = df['pred_sentiment'].map(label_map)

# Drop rows where the LLM output could not be parsed to positive/negative
valid = y_true.notna() & y_pred.notna()
results['Mistral-7B 5-shot'] = metrics(y_true[valid], y_pred[valid])

## Results Table

In [17]:
table = pd.DataFrame(results).T
table.index.name = 'Method'

# Highlight best value per column
table.style \
    .highlight_max(axis=0, props='font-weight: bold') \
    .format('{:.3f}')

,Acc,Mac F1,Neg F1
Method,,,
TF-IDF + LR,0.948,0.893,0.816
SentTransf. + LR,0.950,0.898,0.825
nlptown BERT,0.944,0.901,0.835
CardiffNLP RoBERTa,0.933,0.875,0.790
BART zero-shot,0.952,0.911,0.851
DeBERTa zero-shot,0.950,0.901,0.831
Fine-tuned RoBERTa,0.972,0.944,0.905
Mistral-7B 5-shot,0.958,0.924,0.874
